# Probe Guidance Testing and Visualizations

Begin with visualizing some dataset examples

In [1]:
from pathlib import Path
import copy

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
import torchvision.transforms as T
from scipy.spatial.transform import Rotation

import sys
sys.path.append("/home/jack/code/vjepa2_probe_guidance/vjepa2")
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/jack/code/vjepa2_probe_guidance/vjepa2-probe-guidance-venv/lib/python3.10/site-packages', '/home/jack/code/vjepa2_probe_guidance/vjepa2/src', '/home/jack/code/vjepa2_probe_guidance/vjepa2-probe-guidance-venv/lib/python3.10/site-packages/rerun_sdk', '/home/jack/code/vjepa2_probe_guidance/vjepa2']


In [2]:
from app.vjepa_ll_probe_guidance.utils import init_video_model
from app.vjepa_droid.transforms import make_transforms
from utils.mpc_utils import (
    compute_new_pose,
    poses_to_diff
)

/home/jack/code/vjepa2_probe_guidance/vjepa2-probe-guidance-venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:
encoder, predictor = init_video_model(
        device="cuda:1",
        patch_size=16,
        max_num_frames=512,
        tubelet_size=2,
        model_name="vit_large",
        crop_size=256,
        pred_depth=12,
        pred_num_heads=12,
        pred_embed_dim=768,
        action_embed_dim=6,
        predictor_type="ac",
        pred_is_frame_causal=True,
        use_extrinsics=False,
        use_sdpa=True,
        use_rope=True
    )
target_encoder = copy.deepcopy(encoder)

print(encoder)
print(predictor)

VisionTransformer(
  (patch_embed): PatchEmbed3D(
    (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
  )
  (blocks): ModuleList(
    (0-23): 24 x Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): RoPEAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
      (mlp): MLP(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
)
VisionTransfo

In [4]:
def load_state_dict_with_ddp_fix(model, state_dict):
    new_state_dict = {}
    for k, v in state_dict.items():
        # Remove 'module.' prefix if it exists
        new_key = k.replace("module.", "")
        new_state_dict[new_key] = v

    model.load_state_dict(new_state_dict, strict=True)
    return model

resume_path = os.path.join("/home/jack/code/vjepa2_probe_guidance/vjepa2/outputs/ll_probe_guidance_vitl_3", "best.pt")
if os.path.exists(resume_path):
    print(f"Loading checkpoint from {resume_path}")
    checkpoint = torch.load(resume_path, map_location=torch.device("cpu"))
    encoder = load_state_dict_with_ddp_fix(encoder, checkpoint["encoder"])
    predictor = load_state_dict_with_ddp_fix(predictor, checkpoint["predictor"])
    target_encoder = load_state_dict_with_ddp_fix(target_encoder, checkpoint["target_encoder"])
else:
    print(f"Checkpoint not found at {resume_path}")

Loading checkpoint from /home/jack/code/vjepa2_probe_guidance/vjepa2/outputs/ll_probe_guidance_vitl_3/best.pt


In [5]:
dataset_path = "/home/jack/data/probe_guidance_dataset_june"
test_dir = Path(dataset_path) / "test"

episode_paths = [path for path in test_dir.iterdir()]

episode_states = {}
episode_ultrasound_frames = {}
episode_realsense_frames = {}

for episode_path in episode_paths:
    episode_num = int(str(episode_path).split("/")[-1].split("_")[-1])
    states = np.load(episode_path / "states_6dof.npy")
    #states = states[~np.isnan(states).any(axis=1)]
    episode_states[episode_num] = states
    
    realsense_rgb_dir = episode_path / "realsense_rgb"
    ultrasound_dir = episode_path / "ultrasound"
    
    # Only store paths to the frames, don't keep them all in memory!
    realsense_frame_paths = [path for path in realsense_rgb_dir.iterdir()]
    ultrasound_frame_paths = [path for path in ultrasound_dir.iterdir()]

    episode_realsense_frames[episode_num] = sorted(realsense_frame_paths, key=str)
    episode_ultrasound_frames[episode_num] = sorted(ultrasound_frame_paths, key=str)

In [8]:
episode_nums = list(episode_states.keys())

for episode_num in episode_nums:
    print(f"Episode {episode_num}:")
    print(f"\tstates: {len(episode_states[episode_num])}")
    print(f"\tultrasound frames: {len(episode_ultrasound_frames[episode_num])}")
    print(f"\trealsense frames: {len(episode_realsense_frames[episode_num])}")

Episode 78:
	states: 1891
	ultrasound frames: 1891
	realsense frames: 1891
Episode 18:
	states: 8310
	ultrasound frames: 8310
	realsense frames: 8310
Episode 61:
	states: 1595
	ultrasound frames: 1595
	realsense frames: 1595
Episode 63:
	states: 1539
	ultrasound frames: 1539
	realsense frames: 1539
Episode 47:
	states: 3349
	ultrasound frames: 3349
	realsense frames: 3349
Episode 31:
	states: 2607
	ultrasound frames: 2607
	realsense frames: 2607
Episode 68:
	states: 2285
	ultrasound frames: 2285
	realsense frames: 2285
